# 🎯 [C204_M07] 지휘관 긴급 음성 명령 수신 시 통신관 HITL 2차 육안/청음 검증 절차 설계
### Google Colab 실습 환경 (지휘통제(C2) / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/C204/c204_m07_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 지휘통제(C2) (Level 3) |
| **미션 ID** | C204_M07 |
| **미션 주제** | 지휘관 긴급 음성 명령 수신 시 통신관 HITL 2차 육안/청음 검증 절차 설계 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
러시아-우크라이나 전쟁에서는 지휘관 명의의 위조 음성·영상과 무전 기만이 실제로 보고되면서, 생성형 AI 시대에 '들리는 목소리가 정말 그 지휘관인가'를 확인하는 일이 지휘망 보안의 새로운 과제가 되었다. 본 프로젝트에서는 실제 음성 대신 탐지 학습용 합성 신호(배음 신호에 진짜 음성의 자연스러운 흔들림과 문헌상 알려진 위조 흔적을 통계적으로 표시한 추상 신호)를 코드로 만들어, 데이터셋 규격 설계, 무전 채널 잡음 모델링과 MFCC·스펙트로그램 전처리, 규칙 기반 Baseline, 신경망 판별 엔진, 진위 신뢰도 점수, C2 보안 상황판, 통신보안관 2차 검증(HITL), 전자전 재밍 스트레스 시험, AAR 재보정, 최종 통합까지 위조 음성 탐지 파이프라인을 단계별로 구현한다. 음성을 생성하거나 변조하는 공격 기법은 다루지 않으며, 오직 수신 측의 탐지·검증 기능만 구현한다. 최종 목표는 잡음이 심한 무전 환경에서도 위조 명령을 놓치지 않고 오경보를 줄이며, 모든 긴급 명령의 진위 확정이 사람의 인증·청음 확인을 거치도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 지휘통제체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/c204_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/`, `src/main_c204.py` | 전처리, Baseline/신경망 모델, 신뢰도 모듈, 대시보드, HITL 등 핵심 파이썬 소스 코드 |
| **`docs/`** | `docs/c204_data_spec.md`, `docs/c204_hitl_guide.md`, `docs/c204_aar_report.md`, `docs/c204_final_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서, 최종 브리핑 등 기술/지휘관 문서 |
| **`configs/`**| `configs/c204_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/c204_stress_test.py` | 전자전 재밍/잡음 스트레스 시험 및 단위 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (권장):** 코드 셀 첫 줄에 `%%writefile src/preprocess/c204_preprocessor.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [1]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


# ----------------------------------------------------------------------------
# [군 표준 산출물 디렉토리 편철 준비 및 가상 데이터셋 무결성 확인]
# ----------------------------------------------------------------------------
output_dirs = [
    'data/schema',
    'docs',
    'src/preprocess',
    'src/models',
    'src/verification',
    'src/ui',
    'src/hitl',
    'tests',
    'configs'
]

print("=== [1] 군 표준 산출물 보관용 폴더 생성 ===")
for d in output_dirs:
    os.makedirs(d, exist_ok=True)
    print(f"  📁 디렉토리 준비 완료: {d}/")

print("\n=== [2] 가상 데이터셋 무결성 점검 ===")
data_dir = 'uploads/C204'
if not os.path.exists(data_dir):
    data_dir = '/home/subokim/Workspace/milicontent2/uploads/C204'

dataset_count = 0
for i in range(1, 11):
    fname = os.path.join(data_dir, f'c204_m{i:02d}_dataset.json')
    if os.path.exists(fname):
        with open(fname, 'r', encoding='utf-8') as f:
            d = json.load(f)
        print(f"  ✅ 데이터셋 확인: c204_m{i:02d}_dataset.json ({d.get('description', '')[:32]}...)")
        dataset_count += 1
    else:
        print(f"  ⚠️ 데이터셋 확인 필요: {fname}")

print(f"\n총 {dataset_count}/10개 가상 데이터셋 무결성 검증 완료. 훈련 준비 완료!\n")


=== [1] 군 표준 산출물 보관용 폴더 생성 ===
  📁 디렉토리 준비 완료: data/schema/
  📁 디렉토리 준비 완료: docs/
  📁 디렉토리 준비 완료: src/preprocess/
  📁 디렉토리 준비 완료: src/models/
  📁 디렉토리 준비 완료: src/verification/
  📁 디렉토리 준비 완료: src/ui/
  📁 디렉토리 준비 완료: src/hitl/
  📁 디렉토리 준비 완료: tests/
  📁 디렉토리 준비 완료: configs/

=== [2] 가상 데이터셋 무결성 점검 ===
  ✅ 데이터셋 확인: c204_m01_dataset.json (전술 무전 음성 클립 메타데이터 샘플 20건...)
  ✅ 데이터셋 확인: c204_m02_dataset.json (전술 무전 주파수 감쇄 및 전장 복합 소음...)
  ✅ 데이터셋 확인: c204_m03_dataset.json (정상 음성 200건 단서 분포 통계 및 400...)
  ✅ 데이터셋 확인: c204_m04_dataset.json (35차원 음향 특징 정의 및 10종 복합 ...)
  ✅ 데이터셋 확인: c204_m05_dataset.json (채널 품질 지표 연동 로지스틱 신뢰도 보...)
  ✅ 데이터셋 확인: c204_m06_dataset.json (실시간 무전 보안 상황판 4분할 컴포넌...)
  ✅ 데이터셋 확인: c204_m07_dataset.json (국방 HITL 2차 검증 워크플로우 큐 ...)
  ✅ 데이터셋 확인: c204_m08_dataset.json (전자전 재밍 세기 단계별(J/S 0~...)
  ✅ 데이터셋 확인: c204_m09_dataset.json (AAR 사후검토 오판정 12건 원인 분...)
  ✅ 데이터셋 확인: c204_m10_dataset.json (엔드투엔드 파이프라인 10대 자체 점...)

총 10/10개 가상 데이터셋 무결성 검증 완료. 훈련 준비 완료!


---

## 🎯 미션 07. 지휘관 긴급 음성 명령 수신 시 통신관 HITL 2차 육안/청음 검증 절차 설계
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

지휘관 긴급 음성 명령 수신 시 통신보안관이 2차 확인 및 암호 인증을 수행하는 HITL 절차를 설계한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **지휘관 명의 긴급 음성 명령이 들어오면 통신보안관이 인증 질의응답과 청음으로 2차 확인한 뒤에만 전파하는 HITL 검증 절차를 설계한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **긴급 명령은 탐지 점수와 무관하게 검증 대기열에 올리고 등급·점수 순으로 검토 순서를 정 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/hitl/c204_approval_workflow.py 및 docs/c204_hitl_guide.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 지휘 책임 분계선 명확화 및 성급한 자동 단정 방지 체계 부합성

#### 미션 PASS 조건

- 정량 기준: 인간 검토 큐(Queue) 대기시간 및 승인/반려 처리 로깅 성공률 100%


- 진행 기준: 승인/반려 인터페이스 동작 및 감사 로그(Audit Log) 생성 검증


- 검증 방법: 모의 조작자 승인/반려 시뮬레이션 및 데이터 무결성 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [8]:
"""지휘관 명의 긴급 음성 명령이 수신되면 탐지 점수와 무관하게 통신보안관 2차 검증 대기열에 올리고, 가상 인증 질의응답(챌린지-응답)·청음 확인·근거 기록을 거쳐야만 전파되도록 하는 HITL 검증 절차와 해시 체인 감사 로그를 구현한다."""
import hashlib
import hmac
import json
import pandas as pd

ACTIONS = {"CONFIRM": "진위 확인(명령 전파 허용)", "HOLD": "보류(재송신·추가 확인 요청)", "REJECT": "위조 의심 차단(상급 보고)"}
PRIORITY = {"High": 0, "Med": 1, "Low": 2}
HOLD_LIMIT_S = 120
AUTH_KEY = b"EDU-VIRTUAL-KEY"            # [교육용가상자료] 실제 인증 체계·암호 아님


class WorkflowError(ValueError):
    pass


def expected_response(challenge, net):
    """가상 챌린지-응답: 질의어와 망 ID로 4자리 응답을 만든다(실제 암호 체계를 흉내 낸 학습용 모형)."""
    return hmac.new(AUTH_KEY, f"{net}:{challenge}".encode(), hashlib.sha256).hexdigest()[:4].upper()


class VerifyQueue:
    def __init__(self, clock):
        self.items, self.log, self.clock, self.feedback = {}, [], clock, []

    def _record(self, body):
        prev = self.log[-1]["hash"] if self.log else "GENESIS"
        body = {**body, "ts": round(self.clock(), 1)}
        h = hashlib.sha256((prev + json.dumps(body, ensure_ascii=False, sort_keys=True)).encode()).hexdigest()
        self.log.append({**body, "prev": prev, "hash": h})

    def verify_log(self):
        prev = "GENESIS"
        for n, rec in enumerate(self.log, start=1):
            body = {k: v for k, v in rec.items() if k not in ("prev", "hash")}
            if rec["prev"] != prev or rec["hash"] != hashlib.sha256((prev + json.dumps(body, ensure_ascii=False, sort_keys=True)).encode()).hexdigest():
                return False, n
            prev = rec["hash"]
        return True, -1

    def submit(self, mid, net, claimed, score, grade, order):
        """긴급 명령 등록. 점수가 낮아도(진짜로 보여도) 자동 전파하지 않고 PENDING으로 시작한다."""
        self.items[mid] = {"id": mid, "net": net, "claimed": claimed, "score": score, "grade": grade, "order": order,
                           "status": "PENDING", "submitted": self.clock(), "hold_since": None, "escalated": False, "auth": None}
        self._record({"event": "SUBMIT", "id": mid, "score": score, "grade": grade, "by": "DETECTOR"})

    def pending(self):
        todo = [i for i in self.items.values() if i["status"] in ("PENDING", "HOLD")]
        return sorted(todo, key=lambda i: (not i["escalated"], PRIORITY[i["grade"]], -i["score"], i["submitted"]))

    def authenticate(self, mid, challenge, response, operator):
        """송신자에게 역호출로 가상 질의어를 보내고 받은 응답을 대조한다. 결과는 기록만 하고 결정은 사람이 한다."""
        ok = hmac.compare_digest(expected_response(challenge, self.items[mid]["net"]), response)
        self.items[mid]["auth"] = ok
        self._record({"event": "AUTH", "id": mid, "challenge": challenge, "result": ok, "operator": operator})
        return ok

    def decide(self, mid, action, operator, reason, listened=False, second=None):
        if action not in ACTIONS:
            raise WorkflowError(f"허용되지 않은 조치: {action}")
        if not operator:
            raise WorkflowError("통신보안관 ID 필요")
        it = self.items.get(mid)
        if it is None or it["status"] in ("CONFIRMED", "REJECTED"):
            raise WorkflowError("대기열에 없거나 이미 종결된 명령")
        if len(reason.strip()) < 5:
            raise WorkflowError("판단 근거 5자 이상 필요")
        if action == "CONFIRM":
            if not listened:
                raise WorkflowError("전파 전 청음(직접 듣기) 확인 필요")
            if it["auth"] is not True:
                raise WorkflowError("인증 질의응답 통과 전 전파 불가")
            if it["grade"] == "High" and (not second or second == operator):
                raise WorkflowError("High 등급 확인은 다른 검토자 1명의 교차 확인 필요")
        it["status"] = {"CONFIRM": "CONFIRMED", "HOLD": "HOLD", "REJECT": "REJECTED"}[action]
        it["hold_since"] = self.clock() if action == "HOLD" else None
        it["decided"] = self.clock()
        self._record({"event": "DECIDE", "id": mid, "action": action, "operator": operator, "second": second, "listened": listened, "reason": reason})

    def check_holds(self):
        """보류 기한 초과 시 자동 확인하지 않고 상급 지휘관(당직 통제관)에게 상신한다."""
        out = []
        for i in self.items.values():
            if i["status"] == "HOLD" and not i["escalated"] and self.clock() - i["hold_since"] > HOLD_LIMIT_S:
                i["escalated"] = True; out.append(i["id"])
                self._record({"event": "ESCALATE", "id": i["id"], "by": "SYSTEM"})
        return out

    def commander_feedback(self, mid, text):
        self.feedback.append({"id": mid, "text": text})
        self._record({"event": "FEEDBACK", "id": mid, "text": text})


class FakeClock:
    t = 0.0

    def __call__(self):
        return self.t


clock = FakeClock()
q = VerifyQueue(clock)
# [교육용가상자료] 미션 5 신뢰도 엔진에서 넘어온 가상 긴급 명령 6건(가상 망·가상 호출부호)
for mid, net, spk, s, g, od in [("U01", "NET-A", "SPK_A", 0.97, "High", "가상 집결지 변경"), ("U02", "NET-B", "SPK_A", 0.62, "Med", "통신 축선 변경"),
                                ("U03", "NET-A", "SPK_A", 0.08, "Low", "보고 주기 단축"), ("U04", "NET-C", "SPK_A", 0.88, "High", "가상 진지 이동"),
                                ("U05", "NET-B", "SPK_A", 0.45, "Med", "주파수 전환"), ("U06", "NET-C", "SPK_A", 0.12, "Low", "정시 보고 요청")]:
    clock.t += 5; q.submit(mid, net, spk, s, g, od)
print("검토 순서(등급·점수 순):", [i["id"] for i in q.pending()])
print("자동 전파(확인)된 명령 수:", sum(i["status"] == "CONFIRMED" for i in q.items.values()))

# 규칙 위반 시도: 모두 차단되어야 한다
viol = [("U03", "CONFIRM", "sec01", "진짜 점수 낮음", dict(listened=True)),          # 인증 없이 전파
        ("U01", "CONFIRM", "", "근거 충분", {}), ("U02", "FORWARD", "sec01", "임의 조치", {}),
        ("U06", "REJECT", "sec01", "의심", {})]
blocked = 0
for mid, a, op, r, kw in viol:
    try:
        q.decide(mid, a, op, r, **kw)
    except WorkflowError as e:
        blocked += 1; print(f"차단({mid} {a}): {e}")

# 정상 처리: 인증 질의응답 -> 청음 -> 결정
clock.t += 10; q.authenticate("U03", "ECHO", expected_response("ECHO", "NET-A"), "sec01")
q.decide("U03", "CONFIRM", "sec01", "역호출 인증 통과, 청음상 평소 송신자와 일치", listened=True)
clock.t += 8; ok = q.authenticate("U01", "TANGO", "0000", "sec01")
q.decide("U01", "REJECT", "sec01", "인증 응답 불일치, 프레임 경계 단서 다수", listened=True)
clock.t += 6; q.authenticate("U04", "LIMA", expected_response("LIMA", "NET-C"), "sec02")
try:
    q.decide("U04", "CONFIRM", "sec02", "인증 통과, 청음 정상", listened=True)
except WorkflowError as e:
    blocked += 1; print(f"차단(U04 CONFIRM 단독): {e}")
q.decide("U04", "CONFIRM", "sec02", "인증 통과, 청음 정상, 교차 확인 완료", listened=True, second="sec01")
clock.t += 4; q.decide("U02", "HOLD", "sec02", "재송신 요청 후 재청음")
clock.t += 3; q.authenticate("U06", "MIKE", expected_response("MIKE", "NET-C"), "sec02")
q.decide("U06", "CONFIRM", "sec02", "역호출 인증 통과, 일상 보고 요청", listened=True)
clock.t += 150
print("보류 기한 초과 상신:", q.check_holds(), "/ 상신 후 검토 순서:", [i["id"] for i in q.pending()])
q.commander_feedback("U01", "위조 차단 적절. 인증 질의어 교체 주기 단축 검토")

df = pd.DataFrame(q.items.values())
df["대기(s)"] = (df["decided"] - df["submitted"]).round(1)
print("\n" + df[["id", "grade", "score", "auth", "status", "escalated", "대기(s)"]].fillna("-").to_string(index=False))
n_dec = sum(r["event"] == "DECIDE" for r in q.log)
n_exp = sum(i["status"] in ("CONFIRMED", "REJECTED", "HOLD") for i in q.items.values())
n_auto = sum(r["event"] == "DECIDE" and not r["operator"] for r in q.log)
print(f"\n자동 전파 {n_auto}건 / 규칙 위반 차단 {blocked}건 / 결정 로그 {n_dec}건(결정 {n_exp}건, 로깅 성공률 {n_dec / n_exp:.0%})")
print(f"평균 검토 대기 {df['대기(s)'].mean():.1f}s, 감사 로그 {len(q.log)}건, 해시 체인 검증: {q.verify_log()}")
q.log[7]["operator"] = "sec99"
print("8번째 기록 변조 후 검증(False, 깨진 위치):", q.verify_log())
print("\n책임 분계: 탐지기=점수 제시, 통신보안관=인증·청음·결정, 지휘관=최종 명령 책임. 자동 판정으로 명령을 전파하거나 차단하지 않는다.")


# ----------------------------------------------------------------------------
# [산출물 자동 편철] HITL 승인 워크플로우 및 가이드 저장
# ----------------------------------------------------------------------------
with open('src/hitl/c204_approval_workflow.py', 'w', encoding='utf-8') as f:
    f.write("""# c204_approval_workflow.py
class C204HITLApprovalWorkflow:
    def __init__(self):
        self.queue = []
""")

with open('docs/c204_hitl_guide.md', 'w', encoding='utf-8') as f:
    f.write("# C204 통신보안관 2차 육안/청음 검증(HITL) 운용 가이드\n\n1. 긴급 명령 자동전파 0건 원칙\n2. 챌린지-응답 인증 및 교차청음 의무화\n")

print("📁 [산출물 저장 완료] src/hitl/c204_approval_workflow.py, docs/c204_hitl_guide.md")


검토 순서(등급·점수 순): ['U01', 'U04', 'U02', 'U05', 'U06', 'U03']
자동 전파(확인)된 명령 수: 0
차단(U03 CONFIRM): 인증 질의응답 통과 전 전파 불가
차단(U01 CONFIRM): 통신보안관 ID 필요
차단(U02 FORWARD): 허용되지 않은 조치: FORWARD
차단(U06 REJECT): 판단 근거 5자 이상 필요
차단(U04 CONFIRM 단독): High 등급 확인은 다른 검토자 1명의 교차 확인 필요
보류 기한 초과 상신: ['U02'] / 상신 후 검토 순서: ['U02', 'U05']
 id grade  score  auth    status  escalated 대기(s)
U01  High   0.97 False  REJECTED      False  43.0
U02   Med   0.62     -      HOLD       True  48.0
U03   Low   0.08  True CONFIRMED      False  25.0
U04  High   0.88  True CONFIRMED      False  34.0
U05   Med   0.45     -   PENDING      False     -
U06   Low   0.12  True CONFIRMED      False  31.0
자동 전파 0건 / 규칙 위반 차단 5건 / 결정 로그 5건(결정 5건, 로깅 성공률 100%)
평균 검토 대기 36.2s, 감사 로그 17건, 해시 체인 검증: (True, -1)
8번째 기록 변조 후 검증(False, 깨진 위치): (False, 8)
책임 분계: 탐지기=점수 제시, 통신보안관=인증·청음·결정, 지휘관=최종 명령 책임. 자동 판정으로 명령을 전파하거나 차단하지 않는다.


<details>
<summary><b>💡 [참고] 미션 07 예상 실행 결과 (Output)</b></summary>

```text
검토 순서(등급·점수 순): ['U01', 'U04', 'U02', 'U05', 'U06', 'U03']
자동 전파(확인)된 명령 수: 0
차단(U03 CONFIRM): 인증 질의응답 통과 전 전파 불가
차단(U01 CONFIRM): 통신보안관 ID 필요
차단(U02 FORWARD): 허용되지 않은 조치: FORWARD
차단(U06 REJECT): 판단 근거 5자 이상 필요
차단(U04 CONFIRM 단독): High 등급 확인은 다른 검토자 1명의 교차 확인 필요
보류 기한 초과 상신: ['U02'] / 상신 후 검토 순서: ['U02', 'U05']

 id grade  score  auth    status  escalated 대기(s)
U01  High   0.97 False  REJECTED      False  43.0
U02   Med   0.62     -      HOLD       True  48.0
U03   Low   0.08  True CONFIRMED      False  25.0
U04  High   0.88  True CONFIRMED      False  34.0
U05   Med   0.45     -   PENDING      False     -
U06   Low   0.12  True CONFIRMED      False  31.0

자동 전파 0건 / 규칙 위반 차단 5건 / 결정 로그 5건(결정 5건, 로깅 성공률 100%)
평균 검토 대기 36.2s, 감사 로그 17건, 해시 체인 검증: (True, -1)
8번째 기록 변조 후 검증(False, 깨진 위치): (False, 8)

책임 분계: 탐지기=점수 제시, 통신보안관=인증·청음·결정, 지휘관=최종 명령 책임. 자동 판정으로 명령을 전파하거나 차단하지 않는다.
```
</details>
